In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [49]:
df = pd.read_csv('../data/raw/store-data.csv')

In [50]:
def city_column_formating():
    cities = df['City'].values
    new_cities = []
    for city in cities:
        new_cities.append(city.title().strip())
    df['City'] = new_cities
city_column_formating()

In [51]:
def remove_full_discount():
    df.drop(df[df['Discount'] >= 1].index, inplace=True)
    df.reset_index(drop=True, inplace=True)

remove_full_discount()

In [52]:
def category_column_formating():
    categories = df['Category'].values
    new_categories = []
    for category in categories:
        new_categories.append(category.title().strip())
    df['Category'] = new_categories

category_column_formating()

In [53]:
def state_column_formating():
    states = df['State'].values
    new_states = []
    for state in states:
        new_states.append(state.title().strip())
    df['State'] = new_states

state_column_formating()

In [54]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10049,1281,CA-2016-160815,9/5/2016,9/6/2016,First Class,TR-21325,TOBY RITTER,Consumer,United States,Cedar Rapids,...,52402.0,Central,TEC-PH-10003505,Technology,Phones,Geemarc AmpliPOWER60,278.4000,3.0,0.00,80.7360
10050,7699,CA-2017-151799,12/14/2017,12/18/2017,Standard Class,BF-11170,BEN FERRER,Home Office,United States,Lawrence,...,1841.0,East,OFF-ST-10002790,Office Supplies,Storage,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155
10051,8597,CA-2014-111934,5/5/2014,5/7/2014,First Class,GD-14590,GIULIETTA DORTCH,Corporate,United States,Arlington,...,22204.0,South,OFF-PA-10000474,Office Supplies,Paper,Easy-staple paper,35.4400,1.0,0.00,16.6568
10052,4962,CA-2014-156587,3/7/2014,3/8/2014,First Class,AB-10015,NaN,Consumer,United States,Seattle,...,98103.0,West,FUR-CH-10004477,Furniture,Chairs,"Global Push Button Manager's Chair, Indigo",48.7120,1.0,0.20,5.4801


In [55]:
def drop_negative_quantities():
    df.drop([id for id, boolean in enumerate(df['Quantity'] <= 0) if boolean == True], inplace=True)
    df.reset_index(drop=True, inplace=True)

drop_negative_quantities()

In [56]:
def remove_duplicated_columns():
    df.drop([id for id, boolean in enumerate(df.duplicated()) if boolean == True], inplace=True)
    df.reset_index(drop=True, inplace=True)
remove_duplicated_columns()
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed').dt.strftime('%d/%m/%Y')
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed').dt.strftime('%d/%m/%Y')
remove_duplicated_columns()

In [57]:
# changing between ship-date order-date if order-date > ship-date
mask = pd.to_datetime(df['Order Date'], dayfirst=True) > pd.to_datetime(df['Ship Date'], dayfirst=True)
new_orderDateCol = np.where(mask, df['Ship Date'].values, df['Order Date'].values)
new_shipDateCol = np.where(mask, df['Order Date'].values, df['Ship Date'].values)
df['Order Date'] = new_orderDateCol
df['Ship Date'] = new_shipDateCol

In [58]:
def describe_dataset():
    for column in df.columns:
        nullValues = df[column].isnull()
        duplicatedValues = df[column].duplicated().sum()
        print(f'[+] Column "{column}" Contain: {nullValues.sum()} Null Values => {(nullValues.mean() * 100):.2f}% | Duplicated Values: {duplicatedValues}')

In [59]:
rows, _ = df.shape
df['Row ID'] = np.arange(1, rows + 1, 1)

In [60]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 4995
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8762
[+] Column "Ship Date" Contain: 101 Null Values => 1.01% | Duplicated Values: 8665
[+] Column "Ship Mode" Contain: 299 Null Values => 2.99% | Duplicated Values: 9994
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9206
[+] Column "Customer Name" Contain: 304 Null Values => 3.04% | Duplicated Values: 9188
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 9993
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 9998
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9468
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9950
[+] Column "Postal Code" Contain: 199 Null Values => 1.99% | Duplicated Values: 9293
[+] Column "Region" Contain: 0 

In [61]:
def get_most_close_ship_date(row_id, mean):
    output_arr = np.full(6, mean)
    local_id = row_id
    found_values = 0
    while found_values < 3 and local_id >= 0:
        local_id -= 1
        sub = df[df['Row ID'] == local_id][['Order Date', 'Ship Date']]
        if sub['Ship Date'].isnull().iloc[0]: continue
        output_arr[found_values] = (pd.to_datetime(sub['Ship Date'].iloc[0], dayfirst=True) - pd.to_datetime(sub['Order Date'].iloc[0], dayfirst=True)).total_seconds()
        found_values += 1
    local_id = row_id
    while found_values < 6 and local_id < len(df):
        local_id += 1
        sub = df[df['Row ID'] == local_id][['Order Date', 'Ship Date']]
        if sub['Ship Date'].isnull().iloc[0]: continue
        output_arr[found_values] = (pd.to_datetime(sub['Ship Date'].iloc[0], dayfirst=True) - pd.to_datetime(sub['Order Date'].iloc[0], dayfirst=True)).total_seconds()
        found_values += 1
    return np.divide(output_arr.sum(), 6)



In [62]:
mean_ship_date = np.divide((pd.to_datetime(df['Ship Date'].iloc[0], dayfirst=True) - pd.to_datetime(df['Order Date'].iloc[0], dayfirst=True)).total_seconds(), rows)
for row_id in df['Row ID'][df['Ship Date'].isnull()].values:
    prediction = get_most_close_ship_date(row_id, mean_ship_date)
    target_row = df[df['Row ID'] == row_id]
    order_date = target_row['Order Date'].iloc[0]
    df.loc[target_row.index[0], 'Ship Date'] = pd.to_datetime(pd.to_datetime(order_date, dayfirst=True).timestamp() + prediction).strftime('%d/%m/%Y')

In [63]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 4995
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8762
[+] Column "Ship Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8665
[+] Column "Ship Mode" Contain: 299 Null Values => 2.99% | Duplicated Values: 9994
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9206
[+] Column "Customer Name" Contain: 304 Null Values => 3.04% | Duplicated Values: 9188
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 9993
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 9998
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9468
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9950
[+] Column "Postal Code" Contain: 199 Null Values => 1.99% | Duplicated Values: 9293
[+] Column "Region" Contain: 0 Nu

In [64]:
def get_frequente_ship_mode(row_id, frequente_value):
    output_list = [frequente_value]  * 10
    found_values = 0
    local_row = row_id
    while found_values < 5 and local_row >= 0:
        local_row -= 1
        sub = df['Ship Mode'][df['Row ID'] == local_row]
        if sub.empty: continue
        if pd.isnull(sub.iloc[0]): continue
        output_list[found_values] = sub.iloc[0]
        found_values += 1
    local_row = row_id
    while found_values < 10 and local_row < len(df):
        local_row += 1
        sub = df['Ship Mode'][df['Row ID'] == local_row]
        if sub.empty: continue
        if pd.isnull(sub.iloc[0]): continue
        output_list[found_values] = sub.iloc[0]
        found_values += 1
    return pd.Series(output_list).mode().iloc[0]

In [65]:
mean_ship_mode = df['Ship Mode'].mode().loc[0]
for row_id in df['Row ID'][df['Ship Mode'].isnull()].values:
    prediction = get_frequente_ship_mode(row_id, mean_ship_mode)
    target_row = df[df['Row ID'] == row_id]
    df.loc[target_row.index[0], 'Ship Mode'] = prediction
    # print(f'Order Date: {order_date} | Ship Date (predicted): {pd.to_datetime(pd.to_datetime(order_date, dayfirst=True).timestamp() + prediction, unit='s').strftime('%d/%m/%Y')}')

In [66]:
def clean_customer_name_column():
    customer_id_set = set(df['Customer ID'].values)
    base_customer_name = df['Customer Name'].mode().iloc[0]
    for customer_id in customer_id_set:
        mask = df['Customer ID'] == customer_id
        names = df.loc[mask, 'Customer Name']
        frequente_value = names.mode()
        if frequente_value.empty:
            value = base_customer_name
        else:
            value = frequente_value.iloc[0]
        df.loc[mask, 'Customer Name'] = names.fillna(value)

In [67]:
clean_customer_name_column()

In [68]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 4995
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8762
[+] Column "Ship Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8665
[+] Column "Ship Mode" Contain: 0 Null Values => 0.00% | Duplicated Values: 9995
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9206
[+] Column "Customer Name" Contain: 0 Null Values => 0.00% | Duplicated Values: 9189
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 9993
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 9998
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9468
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9950
[+] Column "Postal Code" Contain: 199 Null Values => 1.99% | Duplicated Values: 9293
[+] Column "Region" Contain: 0 Null V

In [69]:
def clean_postal_code_column():
    cities = set(df['City'].values)
    frequente_postal_code = df['Postal Code'].mode().iloc[0]
    for city in cities:
        mask = df['City'] == city
        postal_codes = df.loc[mask, 'Postal Code']
        frequente_value = postal_codes.mode()
        if frequente_value.empty:
            value = frequente_postal_code
        else:
            value = frequente_value.iloc[0]
        df.loc[mask, 'Postal Code'] = postal_codes.fillna(value)

In [70]:
clean_postal_code_column()

In [71]:
def fill_quantity_with_product_mean():
    products_ids = set(df['Product ID'][df['Quantity'].isna()].values)
    for product_id in products_ids:
        mask = df['Product ID'] == product_id
        quantities = df.loc[mask, 'Quantity']
        if quantities.isna().sum() == len(quantities):
            product_sub_category = df[df['Product ID'] == product_id]['Sub-Category'].iloc[0]
            mean = np.int16(df[df['Sub-Category'] == product_sub_category]['Quantity'].mean())
        else: mean = np.int16(quantities.mean())
        df.loc[mask, 'Quantity'] = quantities.fillna(mean)

fill_quantity_with_product_mean()

In [72]:
df[df['Sub-Category'] == 'Paper']['Quantity'].mean()

np.float64(3.7643636363636364)

In [73]:
df[df['Quantity'].isnull()]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [74]:
describe_dataset()

[+] Column "Row ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 0
[+] Column "Order ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 4995
[+] Column "Order Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8762
[+] Column "Ship Date" Contain: 0 Null Values => 0.00% | Duplicated Values: 8665
[+] Column "Ship Mode" Contain: 0 Null Values => 0.00% | Duplicated Values: 9995
[+] Column "Customer ID" Contain: 0 Null Values => 0.00% | Duplicated Values: 9206
[+] Column "Customer Name" Contain: 0 Null Values => 0.00% | Duplicated Values: 9189
[+] Column "Segment" Contain: 0 Null Values => 0.00% | Duplicated Values: 9993
[+] Column "Country" Contain: 0 Null Values => 0.00% | Duplicated Values: 9998
[+] Column "City" Contain: 0 Null Values => 0.00% | Duplicated Values: 9468
[+] Column "State" Contain: 0 Null Values => 0.00% | Duplicated Values: 9950
[+] Column "Postal Code" Contain: 0 Null Values => 0.00% | Duplicated Values: 9294
[+] Column "Region" Contain: 0 Null Val

In [75]:
def clean_sales_column():
    # rows_ids = set(df['Row ID'][df['Sales'].isna()].values)
    for id in df['Row ID'][df['Sales'].isna()].index:
        row = df.loc[id, :]
        row_quantity = row['Quantity']
        by_quantity = df[(df['Product ID'] == row['Product ID']) & ((df['Quantity'] >= (row_quantity - (row_quantity // 0.3)))) | (df['Quantity'] <= (row_quantity - (row_quantity // 0.3)))]['Sales'].mean()
        # print(by_quantity)
        if not math.isnan(by_quantity):
            new_value = by_quantity
        else:
            sub_category = df[df['Product ID'] == row['Product ID']]['Sub-Category'].iloc[0]
            new_value = df[df['Sub-Category'] == sub_category]['Sales'].mean()
        discount = row['Discount']
        if discount > 0:
            new_value += new_value / discount
        df.loc[id, 'Sales'] = new_value

clean_sales_column()

In [76]:
def fill_initial_product_price_column():
    df['Initial Product Price'] = np.float32((df['Sales'] - (df['Sales'] / 100 *df['Discount'])) / df['Quantity'])

fill_initial_product_price_column()

In [80]:
df['Order ID Transformed']

0       None
1       None
2       None
3       None
4       None
        ... 
9994    None
9995    None
9996    None
9997    None
9998    None
Name: Order ID Transformed, Length: 9999, dtype: object

In [93]:
dataframe_len = len(df)
for column in df.columns:
    if df[column].dtype == 'str':
        new_column_name = f'{column} Transformed'
        df[new_column_name] = [None] * dataframe_len
        for unique_value in df[column].unique():
            mask = df[column] == unique_value
            transformed_value = np.int8(mask)
            df.loc[mask, new_column_name] = pd.Series([transformed_value] * mask.sum(), index=df.index[mask])

In [130]:
# display(pd.Series([np.int8(mask)] * mask.sum(), index=df.index[mask]))
def calculate_z_score(array:np.array):
    mean = array.mean()
    ecart = np.subtract(array, mean)
    ecart_type = np.sqrt(np.divide(np.pow(ecart, 2).sum(), array.shape[0]))
    return np.divide(ecart, ecart_type)

calculate_z_score(np.array([1, 2, 4, 3, 5]))

array([-1.41421356, -0.70710678,  0.70710678,  0.        ,  1.41421356])

In [143]:
pd.Series(calculate_z_score(np.array(df['Profit'].values)) > 3)

0       False
1       False
2       False
3       False
4       False
        ...  
9994    False
9995    False
9996    False
9997    False
9998    False
Length: 9999, dtype: bool

In [ ]:
profits_z_score = calculate_z_score(np.array(df['Profit'].values))
mask = (profits_z_score < -3) | (profits_z_score > 3)

# df[(profits_z_score > 3) | (profits_z_score < 3)]['Profit']

RangeIndex(start=0, stop=9999, step=1)

In [ ]:
df.loc[mask, 'Order Date TEST TWO'] = new_value

ValueError: Must have equal len keys and value when setting with an iterable

In [ ]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Initial Product Price,Order Date Test,Order Date TEST TWO
0,1,CA-2016-152156,THE NEWW VALUE,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136,130.979996,Test Value,THE NEWW VALUE
1,2,CA-2016-152156,THE NEWW VALUE,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820,243.979996,Test Value,THE NEWW VALUE
2,3,CA-2016-138688,12/06/2016,16/06/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714,7.310000,NaN,NaN
3,4,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,190.653687,NaN,NaN
4,5,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164,11.161632,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,9995,CA-2016-128671,11/08/2016,16/08/2016,Standard Class,MT-18070,MICHELLE TRAN,Home Office,United States,Tulsa,...,Office Supplies,Binders,"Premium Transparent Presentation Covers, No Pa...",77.5600,2.0,0.00,35.6776,38.779999,NaN,NaN
9995,9996,CA-2016-160815,05/09/2016,06/09/2016,First Class,TR-21325,TOBY RITTER,Consumer,United States,Cedar Rapids,...,Technology,Phones,Geemarc AmpliPOWER60,278.4000,3.0,0.00,80.7360,92.800003,NaN,NaN
9996,9997,CA-2017-151799,14/12/2017,18/12/2017,Standard Class,BF-11170,BEN FERRER,Home Office,United States,Lawrence,...,Office Supplies,Storage,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155,73.849998,NaN,NaN
9997,9998,CA-2014-111934,05/05/2014,07/05/2014,First Class,GD-14590,GIULIETTA DORTCH,Corporate,United States,Arlington,...,Office Supplies,Paper,Easy-staple paper,35.4400,1.0,0.00,16.6568,35.439999,NaN,NaN
